In [ ]:
import torch

print("GPU disponible :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("Carte graphique :", torch.cuda.get_device_name(0))

In [ ]:
import os
import shutil

os.makedirs("/content/data", exist_ok=True)

total, used, free = shutil.disk_usage("/content")
print(f"Espace disponible : {free / 1024**3:.1f} Go")

In [ ]:
!wget -c \
  https://msd-for-monai.s3-us-west-2.amazonaws.com/Task01_BrainTumour.tar \
  -O /content/data/Task01_BrainTumour.tar

In [ ]:
!tar -xf /content/data/Task01_BrainTumour.tar -C /content/data/

In [ ]:
import os

base = "/content/data/Task01_BrainTumour"

for folder in ["imagesTr", "labelsTr", "imagesTs"]:
    path = os.path.join(base, folder)
    if os.path.isdir(path):
        print(folder, ":", len(os.listdir(path)), "fichiers")
    else:
        print(folder, ": introuvable")

In [ ]:
from pathlib import Path

base = Path("/content/data/Task01_BrainTumour")

for folder in ["imagesTr", "labelsTr", "imagesTs"]:
    path = base / folder
    files = sorted(path.glob("*.nii.gz"))

    print(f"{folder} : {len(files)} fichiers NIfTI")
    print("Exemple :", files[0].name if files else "Aucun")
    print()

images = {p.name for p in (base / "imagesTr").glob("*.nii.gz")}
labels = {p.name for p in (base / "labelsTr").glob("*.nii.gz")}

print("IRM avec annotation :", len(images & labels))
print("IRM sans annotation :", len(images - labels))

In [ ]:
from pathlib import Path

base = Path("/content/data/Task01_BrainTumour")

# Supprimer uniquement les fichiers parasites ._
for folder in ["imagesTr", "labelsTr", "imagesTs"]:
    path = base / folder

    parasites = list(path.glob("._*"))

    for f in parasites:
        if f.is_file():
            f.unlink()

    print(f"{folder} : {len(parasites)} fichiers parasites supprimés")

# Vérifier les vrais fichiers IRM
images = {
    p.name for p in (base / "imagesTr").glob("*.nii.gz")
}

labels = {
    p.name for p in (base / "labelsTr").glob("*.nii.gz")
}

tests = {
    p.name for p in (base / "imagesTs").glob("*.nii.gz")
}

print("\n--- Vérification finale ---")
print("IRM entraînement :", len(images))
print("Masques :", len(labels))
print("IRM test officiel :", len(tests))
print("IRM avec masque :", len(images & labels))
print("IRM sans masque :", len(images - labels))
print("Masques sans IRM :", len(labels - images))

In [ ]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

base = Path("/content/data/Task01_BrainTumour")

# Sélectionner un patient
patient = sorted((base / "imagesTr").glob("*.nii.gz"))[0]

# Charger l'IRM et son annotation
irm = nib.load(str(patient)).get_fdata(dtype=np.float32)
masque = nib.load(str(base / "labelsTr" / patient.name)).get_fdata()

print("Patient :", patient.name)
print("Dimensions IRM :", irm.shape)
print("Dimensions masque :", masque.shape)
print("Classes du masque :", np.unique(masque))

# Sélectionner la coupe contenant le plus de tumeur
slice_idx = np.argmax(np.sum(masque > 0, axis=(0, 1)))

# Afficher les quatre modalités IRM
fig, axes = plt.subplots(1, 5, figsize=(20, 4))

modalites = ["FLAIR", "T1", "T1ce", "T2"]

for i in range(4):
    axes[i].imshow(irm[:, :, slice_idx, i], cmap="gray")
    axes[i].set_title(modalites[i])
    axes[i].axis("off")

# Afficher le masque superposé
axes[4].imshow(irm[:, :, slice_idx, 0], cmap="gray")
axes[4].imshow(
    np.ma.masked_where(masque[:, :, slice_idx] == 0,
                       masque[:, :, slice_idx]),
    cmap="jet",
    alpha=0.5,
    vmin=0,
    vmax=3
)
axes[4].set_title("Segmentation tumorale")
axes[4].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import nibabel as nib
import torch
from pathlib import Path
from sklearn.model_selection import train_test_split

base = Path("/content/data/Task01_BrainTumour")
patients = sorted((base / "imagesTr").glob("*.nii.gz"))

# Échantillon de 40 patients, reproductible
rng = np.random.default_rng(42)
indices = rng.choice(len(patients), size=40, replace=False)
patients = [patients[i] for i in indices]

# Séparation par patient
train_files, temp_files = train_test_split(
    patients, test_size=0.30, random_state=42
)

val_files, test_files = train_test_split(
    temp_files, test_size=0.50, random_state=42
)

print("Patients entraînement :", len(train_files))
print("Patients validation :", len(val_files))
print("Patients test :", len(test_files))

In [ ]:
import numpy as np
import nibabel as nib
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

class BrainTumorDataset(Dataset):
    def __init__(self, patient_files, training=False):
        self.patient_files = patient_files
        self.training = training
        self.slices = []

        for patient_idx, path in enumerate(patient_files):
            mask = np.asarray(
                nib.load(str(base / "labelsTr" / path.name)).dataobj
            )

            for z in range(mask.shape[2]):
                # En entraînement, on garde les coupes avec tumeur
                # et un échantillon de coupes sans tumeur.
                has_tumor = np.any(mask[:, :, z] > 0)

                if not training or has_tumor or z % 5 == 0:
                    self.slices.append((patient_idx, z))

    def __len__(self):
        return len(self.slices)

    def __getitem__(self, idx):
        patient_idx, z = self.slices[idx]
        path = self.patient_files[patient_idx]

        image = np.asarray(
            nib.load(str(path)).dataobj[:, :, z, 0],
            dtype=np.float32
        ).copy()

        mask = np.asarray(
            nib.load(str(base / "labelsTr" / path.name)).dataobj[:, :, z],
            dtype=np.float32
        ).copy()

        # Normalisation des pixels non nuls
        foreground = image[image > 0]

        if foreground.size > 0:
            mean = foreground.mean()
            std = foreground.std() + 1e-6
            image = (image - mean) / std

        image = np.clip(image, -5, 5)
        mask = (mask > 0).astype(np.float32)

        image = torch.from_numpy(image).unsqueeze(0)
        mask = torch.from_numpy(mask).unsqueeze(0)

        image = F.interpolate(
            image.unsqueeze(0),
            size=(128, 128),
            mode="bilinear",
            align_corners=False
        ).squeeze(0)

        mask = F.interpolate(
            mask.unsqueeze(0),
            size=(128, 128),
            mode="nearest"
        ).squeeze(0)

        return image, mask


train_dataset = BrainTumorDataset(train_files, training=True)
val_dataset = BrainTumorDataset(val_files)
test_dataset = BrainTumorDataset(test_files)

train_loader = DataLoader(
    train_dataset, batch_size=16, shuffle=True, num_workers=0
)

val_loader = DataLoader(
    val_dataset, batch_size=16, shuffle=False, num_workers=0
)

print("Coupes entraînement :", len(train_dataset))
print("Coupes validation :", len(val_dataset))
print("Coupes test :", len(test_dataset))

image, mask = train_dataset[0]

print("Dimensions image :", image.shape)
print("Dimensions masque :", mask.shape)
print("Classes du masque :", torch.unique(mask))

In [ ]:
import torch
from monai.networks.nets import UNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UNet(
    spatial_dims=2,
    in_channels=1,
    out_channels=1,
    channels=(16, 32, 64, 128),
    strides=(2, 2, 2),
    num_res_units=2
).to(device)

print("Appareil :", device)
print("Paramètres :", sum(p.numel() for p in model.parameters()))

In [ ]:
%pip install -q monai

In [ ]:
from monai.losses import DiceCELoss

criterion = DiceCELoss(
    sigmoid=True,
    squared_pred=False
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

print("Fonction de perte et optimiseur prêts.")

In [ ]:
from tqdm.auto import tqdm

epochs = 5

train_losses = []
val_losses = []

for epoch in range(epochs):

    model.train()
    total_train_loss = 0

    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):

        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, masks)

        loss.backward()
        optimizer.step()

        total_train_loss += loss.item() * images.size(0)

    avg_train_loss = total_train_loss / len(train_dataset)
    train_losses.append(avg_train_loss)

    model.eval()
    total_val_loss = 0

    with torch.no_grad():
        for images, masks in val_loader:

            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            loss = criterion(outputs, masks)

            total_val_loss += loss.item() * images.size(0)

    avg_val_loss = total_val_loss / len(val_dataset)
    val_losses.append(avg_val_loss)

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f}"
    )

print("Entraînement terminé !")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch

torch.save(model.state_dict(), "/content/mon_modele.pth")

print("✅ Modèle sauvegardé !")

In [ ]:
from google.colab import files

files.download("/content/mon_modele.pth")

In [ ]:
%pip install -q monai nibabel scikit-learn

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
%pip install -q monai


In [ ]:
import torch
from monai.networks.nets import UNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UNet(
    spatial_dims=2,
    in_channels=1,
    out_channels=1,
    channels=(16, 32, 64, 128),
    strides=(2, 2, 2),
    num_res_units=2
).to(device)

model.load_state_dict(
    torch.load(
        "/content/mon_modele.pth",
        map_location=device,
        weights_only=True
    )
)

model.eval()

print("✅ Modèle chargé avec succès !")
print("Appareil :", device)

In [ ]:
from pathlib import Path

base = Path("/content/data/Task01_BrainTumour")

images = list((base / "imagesTr").glob("BRATS_*.nii.gz"))
labels = list((base / "labelsTr").glob("BRATS_*.nii.gz"))

print("IRM disponibles :", len(images))
print("Masques disponibles :", len(labels))

In [ ]:
!mkdir -p /content/data

!wget -c \
https://msd-for-monai.s3-us-west-2.amazonaws.com/Task01_BrainTumour.tar \
-O /content/data/Task01_BrainTumour.tar

In [ ]:
!tar -xf /content/data/Task01_BrainTumour.tar -C /content/data/

In [ ]:
from pathlib import Path

base = Path("/content/data/Task01_BrainTumour")

images = list((base / "imagesTr").glob("BRATS_*.nii.gz"))
labels = list((base / "labelsTr").glob("BRATS_*.nii.gz"))

print("IRM disponibles :", len(images))
print("Masques disponibles :", len(labels))

In [ ]:
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

base = Path("/content/data/Task01_BrainTumour")

# Retrouver les 484 patients
patients = sorted((base / "imagesTr").glob("BRATS_*.nii.gz"))

print("Patients disponibles :", len(patients))

# Reproduire exactement la sélection initiale
rng = np.random.default_rng(42)
indices = rng.choice(len(patients), size=40, replace=False)
patients = [patients[i] for i in indices]

# Même répartition que pendant l'entraînement
train_files, temp_files = train_test_split(
    patients,
    test_size=0.30,
    random_state=42
)

val_files, test_files = train_test_split(
    temp_files,
    test_size=0.50,
    random_state=42
)

print("Entraînement :", len(train_files))
print("Validation :", len(val_files))
print("Test :", len(test_files))

print("\nLes 6 patients de test :")
for patient in test_files:
    print(patient.name)

In [ ]:
import torch
import numpy as np
import nibabel as nib
import torch.nn.functional as F

model.eval()

dice_scores = []

for patient in test_files:

    # Charger l'IRM et le masque réel
    irm = nib.load(str(patient)).get_fdata(dtype=np.float32)

    masque = nib.load(
        str(base / "labelsTr" / patient.name)
    ).get_fdata(dtype=np.float32)

    intersection = 0
    total_pixels = 0

    # Traiter les 155 coupes du patient
    for z in range(irm.shape[2]):

        image = irm[:, :, z, 0]

        # Même normalisation que pendant l'entraînement
        foreground = image[image > 0]

        if foreground.size > 0:
            image = (image - foreground.mean()) / (
                foreground.std() + 1e-6
            )

        image = np.clip(image, -5, 5)

        image = torch.from_numpy(
            image.astype(np.float32)
        )[None, None]

        image = F.interpolate(
            image,
            size=(128, 128),
            mode="bilinear",
            align_corners=False
        ).to(device)

        # Préparer le masque de référence
        target = torch.from_numpy(
            (masque[:, :, z] > 0).astype(np.float32)
        )[None, None]

        target = F.interpolate(
            target,
            size=(128, 128),
            mode="nearest"
        ).to(device)

        # Prédiction du modèle
        with torch.no_grad():
            output = model(image)
            prediction = (
                torch.sigmoid(output) > 0.5
            ).float()

        intersection += (
            prediction * target
        ).sum().item() * 2

        total_pixels += (
            prediction.sum().item()
            + target.sum().item()
        )

    dice = (intersection + 1e-6) / (total_pixels + 1e-6)
    dice_scores.append(dice)

    print(f"{patient.name} : Dice = {dice:.4f}")

print("\n-------------------------")
print(f"DICE MOYEN : {np.mean(dice_scores):.4f}")
print("-------------------------")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import nibabel as nib
import torch
import torch.nn.functional as F

# Sélectionner le patient avec le meilleur Dice
patient = next(p for p in test_files if p.name == "BRATS_039.nii.gz")
irm = nib.load(str(patient)).get_fdata(dtype=np.float32)
masque = nib.load(
    str(base / "labelsTr" / patient.name)
).get_fdata(dtype=np.float32)

# Coupe contenant le plus de pixels tumoraux
z = int(np.argmax((masque > 0).sum(axis=(0, 1))))

image = irm[:, :, z, 0].copy()
foreground = image[image > 0]

if foreground.size > 0:
    image = (image - foreground.mean()) / (
        foreground.std() + 1e-6
    )

image = np.clip(image, -5, 5)

tensor = torch.from_numpy(
    image.astype(np.float32)
)[None, None]

tensor = F.interpolate(
    tensor,
    size=(128, 128),
    mode="bilinear",
    align_corners=False
).to(device)

model.eval()

with torch.no_grad():
    prediction = (
        torch.sigmoid(model(tensor)) > 0.5
    ).float()

prediction = prediction[0, 0].cpu().numpy()

target = torch.from_numpy(
    (masque[:, :, z] > 0).astype(np.float32)
)[None, None]

target = F.interpolate(
    target,
    size=(128, 128),
    mode="nearest"
)[0, 0].numpy()

image_show = tensor[0, 0].cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(image_show, cmap="gray")
axes[0].set_title("IRM FLAIR")

axes[1].imshow(image_show, cmap="gray")
axes[1].imshow(
    np.ma.masked_where(target == 0, target),
    cmap="Greens",
    alpha=0.6,
    vmin=0,
    vmax=1
)
axes[1].set_title("Masque réel (Ground Truth)")

axes[2].imshow(image_show, cmap="gray")
axes[2].imshow(
    np.ma.masked_where(prediction == 0, prediction),
    cmap="Reds",
    alpha=0.6,
    vmin=0,
    vmax=1
)
axes[2].set_title("Prédiction U-Net")

for ax in axes:
    ax.axis("off")

plt.suptitle(f"{patient.name} — Coupe {z}")
plt.tight_layout()
plt.show()

In [ ]:
import json

resultats = {
    "modele": "UNet 2D FLAIR",
    "epochs": 5,
    "patients_train": 28,
    "dice_moyen": 0.7280,
    "dice_par_patient": {
        p.name: float(score)
        for p, score in zip(test_files, dice_scores)
    }
}

with open("/content/resultats_baseline.json", "w") as f:
    json.dump(resultats, f, indent=4)

from google.colab import files
files.download("/content/resultats_baseline.json")

In [ ]:
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

base = Path("/content/data/Task01_BrainTumour")

# Récupérer tous les patients annotés
all_patients = sorted(
    (base / "imagesTr").glob("BRATS_*.nii.gz")
)

# Conserver exactement les mêmes 6 patients de test
test_names = {p.name for p in test_files}

# Exclure aussi les 6 anciens patients de validation
val_names = {p.name for p in val_files}

# Nouveaux patients disponibles pour l'entraînement
available = [
    p for p in all_patients
    if p.name not in test_names
    and p.name not in val_names
]

# Sélectionner 120 patients pour le nouvel entraînement
rng = np.random.default_rng(42)
indices = rng.choice(
    len(available),
    size=120,
    replace=False
)

train_files_v2 = [available[i] for i in indices]

print("Patients entraînement V2 :", len(train_files_v2))
print("Patients validation :", len(val_files))
print("Patients test :", len(test_files))

print(
    "Fuite de données :",
    bool(
        set(train_files_v2) & set(test_files)
        or set(train_files_v2) & set(val_files)
    )
)

In [ ]:
from torch.utils.data import DataLoader

# Vérifier que la classe BrainTumorDataset existe
print("Dataset disponible :", "BrainTumorDataset" in globals())

In [ ]:
# Nouveau dataset avec 120 patients
train_dataset_v2 = BrainTumorDataset(
    train_files_v2,
    training=True
)

# Validation avec les 6 mêmes patients
val_dataset_v2 = BrainTumorDataset(
    val_files,
    training=False
)

# Test avec les 6 mêmes patients
test_dataset_v2 = BrainTumorDataset(
    test_files,
    training=False
)

# DataLoaders
train_loader_v2 = DataLoader(
    train_dataset_v2,
    batch_size=16,
    shuffle=True,
    num_workers=0
)

val_loader_v2 = DataLoader(
    val_dataset_v2,
    batch_size=16,
    shuffle=False,
    num_workers=0
)

print("Coupes entraînement V2 :", len(train_dataset_v2))
print("Coupes validation :", len(val_dataset_v2))
print("Coupes test :", len(test_dataset_v2))

images, masks = next(iter(train_loader_v2))

print("Images :", images.shape)
print("Masques :", masks.shape)

In [ ]:
import numpy as np
import nibabel as nib
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

class BrainTumorDataset(Dataset):

    def __init__(self, patient_files, training=False):
        self.patient_files = patient_files
        self.training = training
        self.slices = []

        for patient_idx, path in enumerate(patient_files):
            mask = np.asarray(
                nib.load(str(base / "labelsTr" / path.name)).dataobj
            )

            for z in range(mask.shape[2]):
                has_tumor = np.any(mask[:, :, z] > 0)

                if not training or has_tumor or z % 5 == 0:
                    self.slices.append((patient_idx, z))

    def __len__(self):
        return len(self.slices)

    def __getitem__(self, idx):
        patient_idx, z = self.slices[idx]
        path = self.patient_files[patient_idx]

        image = np.asarray(
            nib.load(str(path)).dataobj[:, :, z, 0],
            dtype=np.float32
        ).copy()

        mask = np.asarray(
            nib.load(str(base / "labelsTr" / path.name)).dataobj[:, :, z],
            dtype=np.float32
        ).copy()

        foreground = image[image > 0]

        if foreground.size > 0:
            mean = foreground.mean()
            std = foreground.std() + 1e-6
            image = (image - mean) / std

        image = np.clip(image, -5, 5)
        mask = (mask > 0).astype(np.float32)

        image = torch.from_numpy(image).unsqueeze(0)
        mask = torch.from_numpy(mask).unsqueeze(0)

        image = F.interpolate(
            image.unsqueeze(0),
            size=(128, 128),
            mode="bilinear",
            align_corners=False
        ).squeeze(0)

        mask = F.interpolate(
            mask.unsqueeze(0),
            size=(128, 128),
            mode="nearest"
        ).squeeze(0)

        return image, mask

print("✅ BrainTumorDataset disponible !")

In [ ]:
import torch
from monai.networks.nets import UNet
from monai.losses import DiceCELoss

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Recréer l'architecture U-Net
model_v2 = UNet(
    spatial_dims=2,
    in_channels=1,
    out_channels=1,
    channels=(16, 32, 64, 128),
    strides=(2, 2, 2),
    num_res_units=2
).to(device)

# Reprendre les paramètres du premier modèle
model_v2.load_state_dict(model.state_dict())

# Fonction de perte
criterion_v2 = DiceCELoss(sigmoid=True)

# Optimiseur
optimizer_v2 = torch.optim.Adam(
    model_v2.parameters(),
    lr=0.0005
)

print("✅ U-Net V2 prêt !")
print("Appareil :", device)
print("Patients entraînement :", len(train_files_v2))

In [ ]:
import torch
from tqdm.auto import tqdm

epochs_v2 = 10

train_losses_v2 = []
val_losses_v2 = []

best_val_loss = float("inf")

for epoch in range(epochs_v2):

    # ENTRAÎNEMENT
    model_v2.train()
    total_train_loss = 0.0

    for images, masks in tqdm(
        train_loader_v2,
        desc=f"Epoch {epoch+1}/{epochs_v2}"
    ):

        images = images.to(device)
        masks = masks.to(device)

        optimizer_v2.zero_grad()

        outputs = model_v2(images)
        loss = criterion_v2(outputs, masks)

        loss.backward()
        optimizer_v2.step()

        total_train_loss += loss.item() * images.size(0)

    avg_train_loss = total_train_loss / len(train_dataset_v2)
    train_losses_v2.append(avg_train_loss)

    # VALIDATION
    model_v2.eval()
    total_val_loss = 0.0

    with torch.no_grad():
        for images, masks in val_loader_v2:

            images = images.to(device)
            masks = masks.to(device)

            outputs = model_v2(images)
            loss = criterion_v2(outputs, masks)

            total_val_loss += loss.item() * images.size(0)

    avg_val_loss = total_val_loss / len(val_dataset_v2)
    val_losses_v2.append(avg_val_loss)

    print(
        f"Epoch {epoch+1}/{epochs_v2} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f}"
    )

    # SAUVEGARDER LE MEILLEUR MODÈLE
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss

        torch.save(
            model_v2.state_dict(),
            "/content/unet_v2_best.pth"
        )

        print("✅ Meilleur modèle V2 sauvegardé !")

print("\n✅ Entraînement V2 terminé !")

In [ ]:
from google.colab import files

files.download("/content/unet_v2_best.pth")

In [ ]:
import torch
from monai.networks.nets import UNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Recréer le modèle
model_v2 = UNet(
    spatial_dims=2,
    in_channels=1,
    out_channels=1,
    channels=(16, 32, 64, 128),
    strides=(2, 2, 2),
    num_res_units=2
).to(device)

# Charger les meilleurs poids sauvegardés
model_v2.load_state_dict(
    torch.load(
        "/content/unet_v2_best.pth",
        map_location=device,
        weights_only=True
    )
)

model_v2.eval()

print("✅ Meilleur modèle V2 chargé !")
print("GPU :", device)

In [ ]:
import numpy as np
import nibabel as nib
import torch.nn.functional as F

dice_scores_v2 = []

model_v2.eval()

with torch.no_grad():

    for patient in test_files:

        irm = nib.load(str(patient)).get_fdata(dtype=np.float32)

        masque = nib.load(
            str(base / "labelsTr" / patient.name)
        ).get_fdata(dtype=np.float32)

        intersection = 0.0
        total_pixels = 0.0

        for z in range(irm.shape[2]):

            image = irm[:, :, z, 0].copy()

            # Même prétraitement que V1 et V2
            foreground = image[image > 0]

            if foreground.size > 0:
                image = (image - foreground.mean()) / (
                    foreground.std() + 1e-6
                )

            image = np.clip(image, -5, 5)

            image = torch.from_numpy(
                image.astype(np.float32)
            )[None, None]

            image = F.interpolate(
                image,
                size=(128, 128),
                mode="bilinear",
                align_corners=False
            ).to(device)

            target = torch.from_numpy(
                (masque[:, :, z] > 0).astype(np.float32)
            )[None, None]

            target = F.interpolate(
                target,
                size=(128, 128),
                mode="nearest"
            ).to(device)

            output = model_v2(image)

            prediction = (
                torch.sigmoid(output) > 0.5
            ).float()

            intersection += 2 * (
                prediction * target
            ).sum().item()

            total_pixels += (
                prediction.sum().item()
                + target.sum().item()
            )

        dice = (intersection + 1e-6) / (total_pixels + 1e-6)

        dice_scores_v2.append(dice)

        print(f"{patient.name} : Dice = {dice:.4f}")

dice_moyen_v2 = float(np.mean(dice_scores_v2))

print("\n--------------------------")
print(f"DICE V1 : 0.7280")
print(f"DICE V2 : {dice_moyen_v2:.4f}")
print(f"DIFFÉRENCE : {dice_moyen_v2 - 0.7280:+.4f}")
print("--------------------------")

In [ ]:
from google.colab import files

files.download("/content/unet_v2_best.pth")

In [ ]:
import json
from google.colab import files

resultats = {
    "modele": "U-Net 2D FLAIR V2",
    "dice_moyen": dice_moyen_v2,
    "scores_par_patient": {
        p.name: float(s)
        for p, s in zip(test_files, dice_scores_v2)
    },
    "patients_entrainement": 120
}

with open("/content/resultats_v2.json", "w") as f:
    json.dump(resultats, f, indent=4)

files.download("/content/resultats_v2.json")

In [ ]:
import numpy as np
import nibabel as nib
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Sélectionner le patient
patient = next(
    p for p in test_files
    if p.name == "BRATS_039.nii.gz"
)

# Charger l'IRM et le masque
irm = nib.load(str(patient)).get_fdata(dtype=np.float32)
masque = nib.load(
    str(base / "labelsTr" / patient.name)
).get_fdata(dtype=np.float32)

# Choisir la coupe contenant le plus de tumeur
z = int(np.argmax((masque > 0).sum(axis=(0, 1))))

# Prétraitement identique à l'entraînement
image = irm[:, :, z, 0].copy()
foreground = image[image > 0]

if foreground.size > 0:
    image = (image - foreground.mean()) / (
        foreground.std() + 1e-6
    )

image = np.clip(image, -5, 5)

image_tensor = torch.from_numpy(
    image.astype(np.float32)
)[None, None]

image_tensor = F.interpolate(
    image_tensor,
    size=(128, 128),
    mode="bilinear",
    align_corners=False
).to(device)

# Masque réel
target = torch.from_numpy(
    (masque[:, :, z] > 0).astype(np.float32)
)[None, None]

target = F.interpolate(
    target,
    size=(128, 128),
    mode="nearest"
)[0, 0].numpy()

# Prédictions des deux modèles
model.eval()
model_v2.eval()

with torch.no_grad():
    pred_v1 = (
        torch.sigmoid(model(image_tensor)) > 0.5
    )[0, 0].cpu().numpy()

    pred_v2 = (
        torch.sigmoid(model_v2(image_tensor)) > 0.5
    )[0, 0].cpu().numpy()

# Affichage
image_show = image_tensor[0, 0].cpu().numpy()

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

axes[0].imshow(image_show, cmap="gray")
axes[0].set_title("IRM FLAIR")

axes[1].imshow(image_show, cmap="gray")
axes[1].imshow(
    np.ma.masked_where(target == 0, target),
    cmap="Greens",
    alpha=0.6,
    vmin=0,
    vmax=1
)
axes[1].set_title("Ground Truth")

axes[2].imshow(image_show, cmap="gray")
axes[2].imshow(
    np.ma.masked_where(pred_v1 == 0, pred_v1),
    cmap="Reds",
    alpha=0.6,
    vmin=0,
    vmax=1
)
axes[2].set_title("U-Net V1")

axes[3].imshow(image_show, cmap="gray")
axes[3].imshow(
    np.ma.masked_where(pred_v2 == 0, pred_v2),
    cmap="Blues",
    alpha=0.6,
    vmin=0,
    vmax=1
)
axes[3].set_title("U-Net V2")

for ax in axes:
    ax.axis("off")

plt.suptitle(f"{patient.name} — Coupe {z}")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

def dice_score(pred, target):
    intersection = np.sum(pred * target)
    denominator = np.sum(pred) + np.sum(target)

    if denominator == 0:
        return 1.0

    return 2 * intersection / denominator

dice_v1_slice = dice_score(pred_v1, target)
dice_v2_slice = dice_score(pred_v2, target)

print("Patient :", patient.name)
print("Coupe :", z)
print(f"Dice V1 sur cette coupe : {dice_v1_slice:.4f}")
print(f"Dice V2 sur cette coupe : {dice_v2_slice:.4f}")

In [ ]:
import numpy as np

# Tous les patients annotés
all_patients = sorted(
    (base / "imagesTr").glob("BRATS_*.nii.gz")
)

# Exclure tous les patients déjà utilisés
used_names = {
    p.name
    for p in (
        train_files
        + train_files_v2
        + val_files
        + test_files
    )
}

# Patients jamais utilisés
new_patients = [
    p for p in all_patients
    if p.name not in used_names
]

rng = np.random.default_rng(123)

indices = rng.choice(
    len(new_patients),
    size=20,
    replace=False
)

final_test_files = [new_patients[i] for i in indices]

print("Nouveaux patients de test :", len(final_test_files))
print("Chevauchement avec les anciens :", len(
    {p.name for p in final_test_files} & used_names
))

for p in final_test_files:
    print(p.name)

In [ ]:
import torch
import numpy as np
import nibabel as nib
import torch.nn.functional as F
import pandas as pd
from tqdm.auto import tqdm

model.eval()
model_v2.eval()

def evaluer_patient(patient, modeles):
    # Charger les données
    irm = nib.load(str(patient)).get_fdata(dtype=np.float32)
    masque = nib.load(
        str(base / "labelsTr" / patient.name)
    ).get_fdata(dtype=np.float32)

    # Accumulateurs pour chaque modèle
    resultats = {
        nom: {"intersection": 0.0, "total": 0.0}
        for nom in modeles
    }

    with torch.no_grad():
        for z in range(irm.shape[2]):
            image = irm[:, :, z, 0].copy()

            # Même prétraitement que l'entraînement
            foreground = image[image > 0]

            if foreground.size > 0:
                image = (image - foreground.mean()) / (
                    foreground.std() + 1e-6
                )

            image = np.clip(image, -5, 5)

            tensor = torch.from_numpy(
                image.astype(np.float32)
            )[None, None]

            tensor = F.interpolate(
                tensor,
                size=(128, 128),
                mode="bilinear",
                align_corners=False
            ).to(device)

            target = torch.from_numpy(
                (masque[:, :, z] > 0).astype(np.float32)
            )[None, None]

            target = F.interpolate(
                target,
                size=(128, 128),
                mode="nearest"
            ).to(device)

            for nom, reseau in modeles.items():
                prediction = (
                    torch.sigmoid(reseau(tensor)) > 0.5
                ).float()

                resultats[nom]["intersection"] += (
                    2 * (prediction * target).sum().item()
                )

                resultats[nom]["total"] += (
                    prediction.sum().item()
                    + target.sum().item()
                )

    scores = {}

    for nom, valeurs in resultats.items():
        scores[nom] = (
            valeurs["intersection"] + 1e-6
        ) / (
            valeurs["total"] + 1e-6
        )

    return scores


# Évaluation des deux modèles
resultats_finaux = []

for patient in tqdm(final_test_files, desc="Évaluation finale"):
    scores = evaluer_patient(
        patient,
        {"V1": model, "V2": model_v2}
    )

    resultats_finaux.append({
        "Patient": patient.name,
        "Dice_V1": scores["V1"],
        "Dice_V2": scores["V2"]
    })

df = pd.DataFrame(resultats_finaux)

print("\n--- RÉSULTATS ---")
print(df.to_string(index=False))

print("\n--- MOYENNES ---")
print("Dice V1 :", round(df["Dice_V1"].mean(), 4))
print("Dice V2 :", round(df["Dice_V2"].mean(), 4))
print(
    "Amélioration :",
    round(
        df["Dice_V2"].mean() - df["Dice_V1"].mean(),
        4
    )
)

In [ ]:
from google.colab import files

df.to_csv(
    "/content/comparaison_unet_v1_v2.csv",
    index=False
)

files.download("/content/comparaison_unet_v1_v2.csv")